### 1) Create table of comments with tension categories, embeddings, and LLM summarized/shortened comments

### Embedding Model Context Priming
Context priming aims to establish a controlled semantic baseline by introducing broad, known variations or themes common across the input texts. This strategic exposure helps the model de-emphasize the influence of these general variations within individual embeddings, leading to a clearer representation of each text's specific content and distinct perspective.



In [ ]:
Create or replace table TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.tension_comments_with_cats_and_embeddings_V2  as 

WITH CommentClassifications AS (
  SELECT
    COMMENT_ID,
    TYPE,
    TARGET,
    CONTENT,

    -- Classification for 'Build Faster v. Build Better'
    IFF(
      SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        CONTENT,
        [
          OBJECT_CONSTRUCT('label', 'Build Faster v. Build Better',
                           'description', 'This category focuses on the primary tension between the push to "build back faster" and the goal to "build back better" following a wildfire. The conflict weighs the urgent need for residents to return quickly against implementing enhanced, often slower and costlier, resilience measures for future safety. This core dilemma between speed and improved standards sometimes leads to the more radical question of whether rebuilding should occur at all in the highest-risk areas.',
                           'examples', ARRAY_CONSTRUCT(
                             'Don\'t go crazy on regulations that significantly increase costs to rebuild', 'Reduce onerous building regulations.', 'We cannot rebuild LA without putting measures in place to mitigate these natural disasters in the future', 'This is our opportunity to take a look at what needs to be fixed and also make sure any new builds are equipped for future fires and earthquakes.', 'We should buy out homeowners in these areas and turn unsafe zones back into wilderness. If they want to stay, they should have to pay for the true cost.', 'Don\'t need tax relief as much as streamlining approvals to get businesses back open...', 'While we are at it, we should bury all of the power lines', 'As part of the rebuild, we should widen our sidewalks and bury the powerlines',
                             'We need to get people back in their homes ASAP; bureaucratic red tape on new building codes is just slowing everything down.', 'If we rebuild the same way, we\'ll just burn down again. This is a chance to mandate fire-resistant materials and better urban planning, even if it takes longer.', 'Should we even be allowing homes in that canyon? Maybe some areas are just too dangerous, and we need to rethink development there entirely.', 'The debate over expediency versus thoroughness in rebuilding is tearing our community council apart.'
                           )
          ),
          OBJECT_CONSTRUCT('label', 'Not Build Faster v. Build Better',
                           'description', 'This comment does NOT primarily focus on the tension between building faster and building better',
                           'examples', ARRAY_CONSTRUCT(
                                -- Obvious Non-matches (2) - Fire-related but not the tension
                                'The smoke was really thick for days, my asthma acted up.',
                                'Thank you to the first responders who saved so many homes in our neighborhood.',
                                -- Edge Cases (5)
                                'Rebuilding efforts are underway, and many new homes are going up.', 'The cost of lumber has increased significantly, making rebuilding more expensive.', 'We need to ensure all new constructions meet the current fire codes.', 'Our community needs more temporary housing solutions while people rebuild.', 'It\'s important to consider environmentally friendly building materials for reconstruction.',
                                -- Examples from Other Categories (8)
                                'Should the government take over the power company to ensure better maintenance of lines?', 'Insurance companies are gouging us; we need a public option for fire insurance.',
                                'My neighbor needs to clear their brush; it\'s a hazard to all of us.', 'The county should implement a mandatory defensible space program with inspections.',
                                'How can I apply for disaster relief funds to help with my losses?', 'Renters who were displaced need direct financial assistance immediately.',
                                'I don\'t think we should allow more apartment buildings in these narrow canyons; it\'s an evacuation nightmare.', 'To solve the housing crisis, we should allow duplexes to be built where single-family homes were burned.'
                           )
          )
        ],
        { 'task_description': 'The following comment is from individuals in Los Angeles County affected by recent wildfires (e.g., Eaton Fire, Palisades Fire)... Respond with "Build Faster v. Build Better" if it primarily addresses this tension, or "Not Build Faster v. Build Better" otherwise.' }
      )['label']::VARCHAR = 'Build Faster v. Build Better', 'Build Faster v. Build Better', NULL
    ) AS Cat_BuildFaster_v_BuildBetter,

    -- Classification for 'Public v Private'
    IFF(
      SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        CONTENT,
        [
          OBJECT_CONSTRUCT('label', 'Public v Private',
                           'description', 'Comments where the responsibility for essential infrastructure and services (like power lines, insurance) in fire zones sparks debate. Discussions weigh whether public entities or private companies better manage these systems for safety and affordability. These comments also tackle the key conflict of cost allocation: who ultimately pays for upgrades, maintenance, and coverage – companies, customers, or taxpayers?',
                           'examples', ARRAY_CONSTRUCT(
                             'Make utilities publicly owned', 'Maybe insurance should not be managed by private companies', 'We need to prevent costs from being passed to consumers', 'De-regulate to improve service', 'The insurance commissioner needs to meet with insurance companies', 'Edison should be responsible for maintaining their equipment', 'Who is going to pay for it? The utilities or the taxpayers?', 'Utilities like SCE should pay for burying all of the utility lines', 'We should bury powerlines and taxpayers should not have to pay the bill.',
                             'SCE needs to be held accountable for its infrastructure failures; they shouldn\'t be allowed to just raise rates to cover their upgrade costs.', 'Why isn\'t there a state-backed insurance option for high-fire-risk zones when private insurers are pulling out?', 'The government needs to step in and regulate these utility monopolies more strictly to ensure safety, not just profits.', 'If we make utilities public, will they be more responsive or just bogged down in bureaucracy?'
                           )
          ),
          OBJECT_CONSTRUCT('label', 'Not Public v Private',
                           'description', 'This comment does not primarily focus on the debate regarding public versus private responsibility for essential infrastructure and services...',
                           'examples', ARRAY_CONSTRUCT(
                                -- Obvious Non-matches (2)
                                'I\'m worried about potential mudslides now that the hills are bare.',
                                'The community has really come together to support each other after this tragedy.',
                                -- Edge Cases (5)
                                'The power lines in our neighborhood look very old and need repairs.', 'My insurance premium went up by 30% this year!', 'Edison is taking a long time to restore power to our street.', 'We need better maintenance of infrastructure to prevent future disasters.', 'The state provides a lot of services, but are they always efficient?',
                                -- Examples from Other Categories (8)
                                'Should we rebuild homes in areas that are extremely prone to wildfires, or just focus on speed for now?', 'It\'s taking too long to get building permits approved; we need to rebuild faster!',
                                'Every homeowner must be responsible for creating defensible space around their property.', 'The fire department needs more funding for community-wide fire prevention education programs.',
                                'The criteria for receiving financial aid seem unfair to middle-income families.', 'Small businesses destroyed by the fire need grants, not loans, to get back on their feet.',
                                'Building high-density housing in this area will make evacuations impossible during the next fire.', 'We should allow accessory dwelling units (ADUs) to help with the housing shortage after the fires.'
                           )
          )
        ],
        { 'task_description': 'The following comment is from individuals in Los Angeles County affected by recent wildfires (e.g., Eaton Fire, Palisades Fire)... Respond with "Public v Private" if it primarily addresses this tension, or "Not Public v Private" otherwise.' }
      )['label']::VARCHAR = 'Public v Private', 'Public v Private', NULL
    ) AS Cat_Public_v_Private,

    -- Classification for 'Individual v. Collective Mitigation'
    IFF(
      SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        CONTENT,
        [
          OBJECT_CONSTRUCT('label', 'Individual v. Collective Mitigation',
                           'description', 'Comments debating primary responsibility for fire mitigation: Individual vs. Collective? Many stress homeowner duty – maintain property (defensible space), comply with codes, face potential fines. Others argue individual actions aren\'t enough, demanding collective/government solutions: mandatory standards, government programs, stronger coordinated enforcement for neighborhood safety. Importantly, this category it is about efforts and responsibility for preventing the NEXT fire. ',
                           'examples', ARRAY_CONSTRUCT(
                             'We need to enforce existing code and fine people who don\'t maintain their property', 'Homeowners need to create defensible space around their homes', 'Need community coordination for fire prevention', 'Make fire mitigation mandatory or offer financial incentives', 'Fire prevention is a shared responsibility between individuals and government',
                             'My neighbor\'s overgrown yard is a fire hazard to the whole block. There need to be stricter, community-wide enforcement of clearance.', 'It\'s not enough for individual homeowners to clear brush; we need large-scale fuel breaks managed by the county or state.', 'Some folks can\'t afford to do all the mitigation work. We need grants or public programs to help everyone protect their property and the community.', 'Instead of just fining people, how about community workshops and resources to educate on effective fire prevention techniques?', 'We need more firebreaks in the natural area above the neighborhood', 'We should plant more native plants that are more fire resistant.'
                           )
          ),
          OBJECT_CONSTRUCT('label', 'Not Individual v. Collective Mitigation',
                           'description', 'This comment does not primarily focus on the debate between individual versus collective responsibility for fire mitigation. They may be talking about the cleanup of the previous fire or anything not related to fire mitigation or prevention',
                           'examples', ARRAY_CONSTRUCT(
                                -- Obvious Non-matches (2)
                                'Where can I find information on how to clean up ash safely from my property?',
                                'It\'s heartbreaking to see all the wildlife displaced by the fire.',
                                -- Edge Cases (5)
                                'I spent all weekend clearing brush around my house.', 'The fire department did an excellent job fighting the fire.', 'We need more fire hydrants in our neighborhood.', 'Fire-resistant landscaping is a good idea for homes in this area.', 'The community is coming together to support those who lost their homes.',
                                -- Examples from Other Categories (8)
                                'Are we going to rebuild with stronger, more fire-resistant codes, or just put things back as they were to save time?', 'It is imperative that we don\'t rebuild in the highest-risk burn zones.',
                                'The power company needs to bury its lines, and they shouldn\'t pass all the costs to consumers.', 'Is private insurance failing us in high-risk fire zones? Maybe we need a public option.',
                                'The process for applying for FEMA aid is too confusing for many victims.', 'Who gets priority for financial assistance – homeowners, renters, or businesses?',
                                'Increasing housing density here will overwhelm our roads and water supply, making future fires even more dangerous.', 'We need to build more affordable multi-family housing in the burn footprint to address the housing crisis.','Debris must be cleared from all properties, even if liens are needed to force cleanup of abandoned lots. Can\'t leave toxic waste sitting around. All utilities need to be underground, no exceptions.','We need strict deadlines and consequences for lot clearing, plus mandatory environmental testing on all properties - burned or not - to protect our community\'s safety.','Need better coordination with homeowners on debris cleanup and mandatory soil testing to ensure proper environmental recovery.'
                           )
          )
        ],
        { 'task_description': 'The following comment is from individuals in Los Angeles County affected by recent wildfires (e.g., Eaton Fire, Palisades Fire)... Respond with "Individual v. Collective Mitigation" if it primarily addresses this tension, or "Not Individual v. Collective Mitigation" otherwise.' }
      )['label']::VARCHAR = 'Individual v. Collective Mitigation', 'Individual v. Collective Mitigation', NULL
    ) AS Cat_Individual_v_CollectiveMitigation,

    -- Classification for 'Financial Aid Distribution'
    IFF(
      SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        CONTENT,
        [
          OBJECT_CONSTRUCT('label', 'Financial Aid Distribution',
                           'description', 'Comments about allocation and distribution of financial assistance following the disaster, including debates on who should be prioritized and fairest mechanisms for distributing funds.',
                           'examples', ARRAY_CONSTRUCT(
                             'Money should be going straight to the people who lost homes', 'Renters were the most badly hurt and need direct assistance', 'Supporting local businesses should be a very high priority', 'We need an eviction moratorium for displaced residents...', 'Undocumented residents deserve assistance too', 'The free market can and will resolve these issues without targeted aid', 'I am middle class... SEND GRANTS, PLEASE!', 'Underinsured homeowners can\'t afford the new code requirements...',
                             'The aid application process is too complicated; people who need help the most are struggling to navigate it.', 'Should aid be prioritized for those who lost primary residences, or should second homes also be eligible for the same level of support?', 'There are concerns about fraud in aid distribution. We need transparency and accountability in how these funds are being used.', 'What about long-term recovery funds? People will need support for more than just the immediate aftermath.'
                           )
          ),
          OBJECT_CONSTRUCT('label', 'Not Financial Aid Distribution',
                           'description', 'This comment does not primarily focus on the allocation and distribution of financial assistance post-disaster...',
                           'examples', ARRAY_CONSTRUCT(
                                -- Obvious Non-matches (2)
                                'Does anyone know when we will be allowed back into the evacuation zones?',
                                'The Red Cross has set up a shelter at the high school gym.',
                                -- Edge Cases (5)
                                'Many people lost everything in the fire; it\'s a terrible tragedy.', 'I donated to the Red Cross to help fire victims.', 'The economic impact of this fire on our town will be huge.', 'It\'s going to cost a lot to rebuild my home.', 'Local charities are providing food and shelter.',
                                -- Examples from Other Categories (8)
                                'We need to streamline the permitting process so people can rebuild faster, even if it means delaying some new code adoptions.', 'Building back better means using superior materials and designs, which will take more time and money but is worth it for future safety.',
                                'Should taxpayers foot the bill for utility companies to upgrade their infrastructure, or should shareholders bear that cost?', 'The state needs to regulate insurance companies more heavily to prevent them from dropping coverage in fire-prone areas.',
                                'If homeowners don\'t maintain their properties, they should be fined; it\'s not just their problem, it\'s a community risk.', 'The county needs a program to help elderly residents clear their properties because they can\'t do it themselves.',
                                'Allowing developers to build dense housing projects in these canyons is irresponsible from a fire safety perspective.', 'We must find ways to increase housing supply in the region, perhaps by upzoning areas that were less affected by the fire.'
                           )
          )
        ],
        { 'task_description': 'The following comment is from individuals in Los Angeles County affected by recent wildfires (e.g., Eaton Fire, Palisades Fire)... Respond with "Financial Aid Distribution" if it primarily addresses this tension, or "Not Financial Aid Distribution" otherwise.' }
      )['label']::VARCHAR = 'Financial Aid Distribution', 'Financial Aid Distribution', NULL
    ) AS Cat_FinancialAidDistribution,

    -- Classification for 'Density v. Safety'
    IFF(
      SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
        CONTENT,
        [
          OBJECT_CONSTRUCT('label', 'Density v. Safety',
                           'description', 'The core conflict: Housing density versus safety in post-wildfire development. Comments favoring higher density argue it\'s needed to meet housing demand. Opposing comments stress safety risks: increased density potentially worsening fire danger, straining evacuation routes, changes neighborhood character, and overwhelming infrastructure in vulnerable areas, questioning how to balance development needs with safety imperatives.',
                           'examples', ARRAY_CONSTRUCT(
                             'Allow for higher density housing to address the housing shortage', 'Build more multi-family dwellings in the rebuilding process', 'Do NOT add housing density in fire-prone areas', 'ADUs should not be allowed in fire prone areas', 'We need wider roads for evacuation before adding more housing',
                             'Building more apartments in these hillsides is just asking for trouble when the next fire hits. Evacuation routes are already choked.', 'We have a housing crisis, and fire-rebuilt areas are an opportunity to increase density responsibly with modern safety standards.', 'Increased density will strain our already limited water resources, which is critical during a fire.', 'The character of our neighborhood will be ruined if they allow developers to just pack in as many units as possible after the fire.'
                           )
          ),
          OBJECT_CONSTRUCT('label', 'Not Density v. Safety',
                           'description', 'This comment does not primarily focus on the conflict between housing density and safety in post-wildfire development...',
                           'examples', ARRAY_CONSTRUCT(
                                -- Obvious Non-matches (2)
                                'I lost all my family photos; it\'s devastating.',
                                'The smell of smoke still lingers in the air even weeks later.',
                                -- Edge Cases (5)
                                'There\'s a lot of new construction happening in town.', 'Evacuation routes were completely gridlocked during the fire.', 'We need more affordable housing in our county.', 'The fire destroyed several apartment buildings.', 'Our neighborhood has a unique character that we want to preserve during rebuilding.',
                                -- Examples from Other Categories (8)
                                'The priority should be getting people back into their homes quickly, even if it means using standard building plans.', 'Let\'s use this opportunity to rebuild with the latest fire-resistant technologies and community design, no matter how long it takes.',
                                'Who should pay for burying the power lines – the utility company or the taxpayers?', 'If private insurance won\'t cover us, we need a government-backed insurance program.',
                                'It\'s up to each homeowner to make their property fire-safe; the government can\'t do everything.', 'We need a coordinated, county-wide effort to manage fuel loads on public and private lands.',
                                'The distribution of aid money seems to be favoring businesses over individual homeowners, which isn\'t right.', 'How can we ensure that financial assistance reaches the most vulnerable populations quickly and fairly?'
                           )
          )
        ],
        { 'task_description': 'The following comment is from individuals in Los Angeles County affected by recent wildfires (e.g., Eaton Fire, Palisades Fire)... Respond with "Density v. Safety" if it primarily addresses this tension, or "Not Density v. Safety" otherwise.' }
      )['label']::VARCHAR = 'Density v. Safety', 'Density v. Safety', NULL
    ) AS Cat_Density_v_Safety

  FROM TRANSFORM_ENGCA_PRD.ANALYTICS.STG_COMMENTS 
),
UnionedClassifiedComments AS (
    SELECT cc.COMMENT_ID, cc.TYPE, cc.TARGET, cc.CONTENT, 'Build Faster v. Build Better' AS tension_category
    FROM CommentClassifications cc
    WHERE cc.Cat_BuildFaster_v_BuildBetter IS NOT NULL

    UNION ALL

    SELECT cc.COMMENT_ID, cc.TYPE, cc.TARGET, cc.CONTENT, 'Public v Private' AS tension_category
    FROM CommentClassifications cc
    WHERE cc.Cat_Public_v_Private IS NOT NULL

    UNION ALL

    SELECT cc.COMMENT_ID, cc.TYPE, cc.TARGET, cc.CONTENT, 'Individual v. Collective Mitigation' AS tension_category
    FROM CommentClassifications cc
    WHERE cc.Cat_Individual_v_CollectiveMitigation IS NOT NULL

    UNION ALL

    SELECT cc.COMMENT_ID, cc.TYPE, cc.TARGET, cc.CONTENT, 'Financial Aid Distribution' AS tension_category
    FROM CommentClassifications cc
    WHERE cc.Cat_FinancialAidDistribution IS NOT NULL

    UNION ALL

    SELECT cc.COMMENT_ID, cc.TYPE, cc.TARGET, cc.CONTENT, 'Density v. Safety' AS tension_category
    FROM CommentClassifications cc
    WHERE cc.Cat_Density_v_Safety IS NOT NULL
)
SELECT
    ucc.COMMENT_ID,
    ucc.TYPE,
    ucc.TARGET,
    ucc.CONTENT,
    ucc.tension_category,

    CASE
    WHEN ucc.tension_category = 'Density v. Safety' THEN
        'Tension: Housing Density vs. Safety. Focus: The conflict between increasing housing density to meet demand post-wildfire versus ensuring safety. Pro-density arguments highlight housing needs. Safety-focused arguments emphasize risks like worsened fire danger, strained evacuation/infrastructure, and altered neighborhood character in vulnerable, high-risk rebuilding zones.'

    WHEN ucc.tension_category = 'Individual v. Collective Mitigation' THEN
        'Tension: Individual vs. Collective Fire Mitigation. Focus: The debate over primary responsibility for wildfire mitigation. One side emphasizes individual homeowner duties (defensible space, code compliance, personal preparedness). The other advocates for broader collective/governmental action (mandatory standards, public programs, coordinated enforcement) due to the interconnected nature of community-wide risk.'

    WHEN ucc.tension_category = 'Financial Aid Distribution' THEN
        'Tension: Equitable Financial Aid Distribution. Focus: The debate surrounding fair allocation of post-disaster financial assistance. Key questions involve who should be prioritized (e.g., homeowners vs. renters, insured vs. underinsured, income levels), the mechanisms for distribution, and defining "equitable" outcomes in providing relief funds.'

    WHEN ucc.tension_category = 'Public v Private' THEN
        'Tension: Public vs. Private Responsibility for Infrastructure. Focus: The conflict over whether public entities or private companies should primarily manage and fund essential infrastructure (e.g., power lines, water systems, insurance provision) in fire-prone areas. This includes the core debate on cost allocation: who bears the financial burden for upgrades, maintenance, and risk coverage – private companies, utility customers, or general taxpayers?'

    WHEN ucc.tension_category = 'Build Faster v. Build Better' THEN
        'Tension: Build Faster vs. Build Better. Focus: The dilemma between the urgent need for residents to return to their homes quickly against implementing enhanced, often slower and costlier, resilience measures for future safety. This core dilemma between speed and improved standards sometimes leads to the more radical question of whether rebuilding should occur at all in the highest-risk areas..'

END AS tension_cat_definition_text,

-- Constructs a conditional context primer by combining a universal wildfire introduction
-- with the specific tension category definition, then appends the survey target and
-- original comment for nuanced semantic embedding relevant to that tension.
(
  -- Universal Wildfire Context Primer 
  'This comment is from an individual impacted by the January 2025 Southern California wildfires. These devastating events included the human-caused Palisades Fire in Pacific Palisades and the suspected utility-linked Eaton Fire in Altadena (implicating entities like SoCal Edison/SCE). Fueled by severe Santa Ana winds and prolonged dry conditions, these fires resulted in fatalities, mass evacuations, and the widespread destruction of thousands of homes and community structures. The aftermath brought significant economic disruption, profound emotional trauma, and major concerns regarding debris removal and hazardous materials. Common overarching themes in survivor feedback include a severe insurance crisis, urgent calls for infrastructure upgrades (notably undergrounding power lines), complex rebuilding challenges, questions about government and utility accountability, and debates surrounding financial aid distribution. This specific comment is further viewed through the particular area of community tension detailed next. \n' ||

  -- Dynamic Tension Category Definition (from the CASE statement)
  tension_cat_definition_text ||

  -- Framing the TARGET and CONTENT
  'User Comment: Regarding the survey topic of "' || TARGET || '", the affected individual stated: "' ||
  CONTENT || '"'
) AS primed_comment_for_embedding,



  
SNOWFLAKE.CORTEX.EMBED_TEXT_1024('voyage-multilingual-2', primed_comment_for_embedding ) as primed_comment_embedding,



SNOWFLAKE.CORTEX.COMPLETE(
  'claude-3-5-sonnet', 
  ARRAY_CONSTRUCT(    
    OBJECT_CONSTRUCT(
      'role', 'system',
      'content', 'You will be provided a user comment from a survey, which will be framed around a specific topic or issue. These comments are from people impacted by recent wildfires in Los Angeles County (specifically the January 2025 Eaton Fire in Altadena and the Palisades Fire in Pacific Palisades).

TASK: Your objective is to create a succinct summary of the user\'s core message or main points *specifically concerning the stated topic/issue*. Focus on capturing the essence of their perspective on this topic, particularly as it relates to the wildfire\'s impact, recovery, prevention, or associated challenges. Your response must be ONLY the summarized comment itself, and it must be 30 words or less. The summary must still sound like it is in the user\'s own voice.'
    ),
    OBJECT_CONSTRUCT(
      'role', 'user',
      'content', 'Regarding the issue of ' || TARGET || ', an affected individual offered this perspective: "' || CONTENT || '"'
    )
  ),
  OBJECT_CONSTRUCT(   
    'temperature', 0.05,
    'max_tokens', 40 -- Approx. 30 words is ~40 tokens.
  )
)['choices'][0]['messages']::varchar AS llm_summary,


  
  
FROM UnionedClassifiedComments ucc
WHERE ucc.tension_category IS NOT NULL 
  AND TRIM(ucc.tension_category) != '' ;

  select * from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.tension_comments_with_cats_and_embeddings limit 10


For future models, I suggest adding the following context priming to account for the variance introduced by the survey targets. 

```
-- Universal Wildfire Context Primer (incorporating all survey target categories)
 'The survey from which this comment is drawn invited perspectives across a range of critical recovery areas, including: Housing & rebuilding; Financial & legal assistance; Infrastructure & utilities restoration; Emergency planning & community safety; Emergency communication; Debris removal & environmental recovery; Economic recovery & small business support; Prioritization of recovery themes; Wildfire prevention prioritization & accountability; Climate & community resilience; and Emotional & mental health support. ',
```

### Get data ready for topic modeling

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import numpy as np
import os
os.environ["NUMBA_CACHE_DIR"] = "/tmp"
os.environ["TRANSFORMERS_CACHE"] = "/tmp"
os.environ["TRANSFORMERS_CACHE"] = "/tmp"

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

cmnt_embeddings_df = session.sql('''
Select 
COMMENT_ID
, TYPE
, TARGET
,  CONTENT
, tension_category
 ,LLM_SUMMARY
, primed_comment_embedding
from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.tension_comments_with_cats_and_embeddings
where tension_category is not null
;
''').to_pandas()

# comment_embeddings = np.array(cmnt_embeddings_df['CMT_EMBEDDING'].tolist())
# summary_embeddings = np.array(cmnt_embeddings_df['SUMMARY_EMBEDDING'].tolist())
full_embedding = np.array(cmnt_embeddings_df['PRIMED_COMMENT_EMBEDDING'].tolist())
docs = np.array(cmnt_embeddings_df['CONTENT'].tolist())
docs_summary = np.array(cmnt_embeddings_df['LLM_SUMMARY'].tolist())
categories = cmnt_embeddings_df['TENSION_CATEGORY'].tolist()

# Conduct Topic Modeling

In [ ]:
from sklearn.cluster import KMeans
from umap import UMAP
from bertopic import BERTopic
import pandas as pd
import numpy as np
import json
import os
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
    
# Create a custom CountVectorizer with stop words removal and df parameters
vectorizer_model = CountVectorizer(
    stop_words='english',           # Remove English stop words
    min_df=0.01,                    # Ignore terms that appear in less than X% of documents
    max_df=0.5,                     # Ignore terms that appear in more than X% of documents
    ngram_range=(1, 2)              # Consider both unigrams and bigrams for more meaningful phrases
)

# Dictionary to store results
category_topics = {}
all_topics_df = pd.DataFrame()  # Will hold all topics across categories

# --- UMAP Parameters for Topic Modeling ---
# Category-specific UMAP parameters for the first UMAP (topic modeling)
umap_topic_model_params = {
    'Density v. Safety': {
        'n_neighbors': 10,
        'n_components': 10 
    },
    'Build Faster v. Build Better': {
        'n_neighbors': 8, 
        'n_components': 3,
    },
    'Individual v. Collective Mitigation': {
        'n_neighbors': 4,
        'n_components': 5,      
        'min_dist': 0.00,
        'random_state': 1
    },
    'Financial Aid Distribution': {
        'n_neighbors': 6,
        'n_components': 3
    },
    'Public v Private': {
        'n_neighbors': 10,
        'n_components': 3
    }
}

# Default UMAP parameters for topic modeling if not specified for a category
default_umap_topic_model_params = {
    'n_neighbors': 8,
    'n_components': 10,      # Default number of components for topic modeling
    'min_dist': 0.05,
    'metric': 'euclidean',
    'random_state': 42
}



# --- HDBSCAN Parameters ---
# HDBSCAN parameters for each category
hdbscan_params = {
    'Density v. Safety': {
        'min_cluster_size': 4,
        'min_samples':2
    },
    'Build Faster v. Build Better': {
        'min_cluster_size': 16,
        'min_samples': 15
    },
    'Individual v. Collective Mitigation': {
        'min_cluster_size': 14,
        'min_samples': 7
    },
    'Financial Aid Distribution': {
        'min_cluster_size':17,
        'min_samples': 17
    },
    'Public v Private': {
        'min_cluster_size': 12,
        'min_samples':10
    }
}

# Default HDBSCAN parameters if not specified for a category
default_hdbscan_params = {
    'min_cluster_size': 8,
    'min_samples': 7,
    'metric': 'euclidean', # HDBSCAN often works on the UMAP output, so euclidean is common here
    'cluster_selection_method': 'eom',
    'prediction_data': True
}

# --- UMAP Parameters for 2D Visualization ---
# Category-specific parameters for the 2D UMAP visualization
umap_visualization_params = {
    'Density v. Safety': {
        'n_neighbors': 50,
        'min_dist': 0.99,
        'target_weight': 0.10
    },
    'Build Faster v. Build Better': {
        'n_neighbors': 5,
        'min_dist': 0.9999999,
        'target_weight': 0.05
    },
    'Individual v. Collective Mitigation': {
        'n_neighbors': 7,
        'min_dist': 0.9,
        'target_weight': 0.9
    },
    'Financial Aid Distribution': {
        'n_neighbors': 20,
        'min_dist': 0.8,
        'target_weight': 0.5
    },
    'Public v Private': {
        'n_neighbors': 10,
        'min_dist': 0.5,
        'target_weight': 0.8
    }
}


# Default UMAP parameters for visualization
default_umap_visualization_params = {
    'n_neighbors': 20,
    'min_dist': 0.8,
    'target_weight': 0.2,
    'metric': 'euclidean',
    'target_metric': 'categorical',
    'random_state': 50
}

# Iterate through each unique category
unique_categories = list(set(categories)) # Ensure 'categories' is defined
print(f"Found {len(unique_categories)} unique categories")

for category in unique_categories:
    # Skip None category
    if category is None or category == 'None':
        print("Skipping None category")
        continue
        
    print(f"\nProcessing category: {category}")
    
    # Filter data for this category
    category_indices = [i for i, cat in enumerate(categories) if cat == category]
    print(f"Processing category: {category} ({len(category_indices)} comments)")
    
    # Skip if too few documents
    min_docs_for_processing = max(default_umap_topic_model_params.get('n_neighbors', 5), 
                                  default_hdbscan_params.get('min_cluster_size', 5), 
                                  10) # Ensure enough samples for UMAP and HDBSCAN
    
    # Adjust min_docs if category specific UMAP params exist
    if category in umap_topic_model_params:
        min_docs_for_processing = max(min_docs_for_processing, umap_topic_model_params[category].get('n_neighbors',5))
    if category in hdbscan_params:
         min_docs_for_processing = max(min_docs_for_processing, hdbscan_params[category].get('min_cluster_size',5))


    if len(category_indices) < min_docs_for_processing:
        print(f"Skipping category '{category}' - not enough documents ({len(category_indices)}). Needs at least {min_docs_for_processing}.")
        continue
    
    # Extract documents and embeddings for this category
    category_docs = np.array(docs_summary)[category_indices] # Ensure docs is indexable like a list or numpy array
    category_embeddings = full_embedding[category_indices]
    category_viz_embeddings = full_embedding[category_indices]
    # category_embeddings = pca_embeddings[category_indices]
    # category_viz_embeddings = pca_embeddings[category_indices]

    # Skip if not enough valid embeddings (already checked by len(category_indices))
    # Convert to numpy array
    category_embeddings = np.array(category_embeddings)
    category_viz_embeddings = np.array(category_viz_embeddings)
    
    # --- Get UMAP parameters for topic modeling for this category ---
    current_umap_params = default_umap_topic_model_params.copy()
    if category in umap_topic_model_params:
        current_umap_params.update(umap_topic_model_params[category])
    
    print(f"Using UMAP parameters for topic modeling in '{category}': "
          f"n_neighbors={current_umap_params['n_neighbors']}, "
          f"n_components={current_umap_params['n_components']}, "
          f"min_dist={current_umap_params['min_dist']}")

    # Create UMAP model with n components for topic modeling
    umap_model = UMAP(
        n_neighbors=current_umap_params['n_neighbors'],
        n_components=current_umap_params['n_components'],
        min_dist=current_umap_params['min_dist'],
        metric=current_umap_params['metric'],
        random_state=current_umap_params['random_state']
    )
    
    # Get HDBSCAN parameters for this category or use defaults
    category_hdbscan_params = default_hdbscan_params.copy()
    if category in hdbscan_params:
        category_hdbscan_params.update(hdbscan_params[category])
        
    print(f"Using HDBSCAN parameters for '{category}': "
          f"min_cluster_size={category_hdbscan_params['min_cluster_size']}, "
          f"min_samples={category_hdbscan_params['min_samples']}")
    
    # Create HDBSCAN clustering model
    hdbscan_model = HDBSCAN(
        min_cluster_size=category_hdbscan_params['min_cluster_size'],
        min_samples=category_hdbscan_params['min_samples'],
        metric=category_hdbscan_params['metric'],
        cluster_selection_method=category_hdbscan_params['cluster_selection_method'],
        prediction_data=category_hdbscan_params['prediction_data']
    )

        
    # Create BERTopic model
    topic_model = BERTopic(
        top_n_words=20,
        hdbscan_model=hdbscan_model,
        umap_model=umap_model,
        vectorizer_model=vectorizer_model,
        verbose=True
    )
    
    # Fit BERTopic with embeddings
    try:
        topics, probs = topic_model.fit_transform(list(category_docs), category_embeddings) # Pass docs as list
    except Exception as e:
        print(f"Error during BERTopic fit_transform for category '{category}': {e}")
        print(f"Number of documents: {len(category_docs)}, Embeddings shape: {category_embeddings.shape}")
        print(f"UMAP params: {current_umap_params}")
        print(f"HDBSCAN params: {category_hdbscan_params}")
        continue # Skip to next category if BERTopic fails

    # Create topic info dataframe
    topic_info = topic_model.get_topic_info()
    print(f"Found {len(topic_info) -1 if not topic_info.empty and topic_info.iloc[0]['Topic'] == -1 else len(topic_info)} actual topics (excluding outliers) for category '{category}'")



    # Count how many documents were assigned to each topic
    unique_topic_ids, topic_doc_counts = np.unique(topics, return_counts=True)
    print("Topic distribution:")
    for topic_id, count in zip(unique_topic_ids, topic_doc_counts):
        if topic_id != -1:
            print(f"  Topic {topic_id}: {count} documents")
    if -1 in unique_topic_ids:
        print(f"  Unassigned (-1): {list(topics).count(-1)} documents")
    else:
        print(f"  Unassigned (-1): 0 documents")

        
    # Add category column
    topic_info['CATEGORY'] = category
    
    # Create a unique prefix for topics
    safe_category = "".join(filter(str.isalnum, str(category)))[:5] if category else "unk" # Make safer prefix
    cat_prefix = f"{safe_category}_{len(all_topics_df)}_" # len(all_topics_df) might not be ideal if parallel, but ok sequentially
    
    topic_info['ORIGINAL_TOPIC'] = topic_info['Topic']
    topic_info['Topic'] = topic_info['Topic'].apply(lambda x: f"{cat_prefix}{x}" if x != -1 else f"{cat_prefix}outlier")
    
    # Add to the combined dataframe
    all_topics_df = pd.concat([all_topics_df, topic_info])
    
    # Create dataframe with documents and their topics
    doc_df = pd.DataFrame({
        'COMMENT_ID': [cmnt_embeddings_df.iloc[idx]['COMMENT_ID'] for idx in category_indices[:len(topics)]],
        'SUMMARY': category_docs[:len(topics)],
        'CATEGORY': category,
        'TOPIC': [f"{cat_prefix}{t}" if t != -1 else f"{cat_prefix}outlier" for t in topics[:len(category_docs)]],
        'ORIGINAL_TOPIC': topics[:len(category_docs)],
        'CONTENT': [cmnt_embeddings_df.iloc[idx]['CONTENT'] for idx in category_indices[:len(topics)]],
    })
    
    # Now create a supervised 2D UMAP using the topic assignments as labels
    print("Creating supervised 2D UMAP using topic assignments as labels...")
    
    # Use the original topic numbers as labels, but set -1 topics to None for supervised UMAP
    topic_labels = np.array(topics[:len(category_docs)])
    
    # Use the original topic numbers (-1 for unassigned) as labels for supervised UMAP
    # Ensure it's an integer array, as UMAP expects this for categorical targets.
    topic_labels_for_umap = np.array(topics[:len(category_docs)], dtype=np.int_) # Using np.int_ or np.int32 etc.
    
    # Get UMAP visualization parameters for this category
    visualization_params = default_umap_visualization_params.copy()
    if category in umap_visualization_params:
        visualization_params.update(umap_visualization_params[category])
    
    print(f"Using UMAP parameters for visualization in '{category}': "
          f"n_neighbors={visualization_params['n_neighbors']}, "
          f"min_dist={visualization_params['min_dist']}, "
          f"target_weight={visualization_params['target_weight']}")
    
    # Create supervised UMAP model for visualization
    umap_model_2d = UMAP(
        n_neighbors=visualization_params['n_neighbors'],
        n_components=2,
        min_dist=visualization_params['min_dist'],
        metric=visualization_params['metric'],
        target_metric=visualization_params['target_metric'],
        target_weight=visualization_params['target_weight'],
        random_state=visualization_params['random_state']
    )
    
    # Handle the case where all documents might be unassigned (-1)
    if np.all(topic_labels == -1):
        print("All documents are unassigned. Using unsupervised UMAP for visualization.")
        umap_2d_embeddings = umap_model_2d.fit_transform(category_viz_embeddings)
    else:
        # Fit supervised UMAP with topic labels
        umap_2d_embeddings = umap_model_2d.fit_transform(category_viz_embeddings, y=topic_labels_for_umap)
    
    # Add UMAP coordinates
    doc_df['UMAP_1'] = [coord[0] for coord in umap_2d_embeddings[:len(doc_df)]]
    doc_df['UMAP_2'] = [coord[1] for coord in umap_2d_embeddings[:len(doc_df)]]
    
    # Store in category_topics dictionary
    category_topics[category] = {
        'model': topic_model,
        'doc_df': doc_df,
        'topic_info': topic_info
    }

# Convert all topic info to strings for Snowflake
for col in all_topics_df.columns:
    if all_topics_df[col].dtype == 'object':
        all_topics_df[col] = all_topics_df[col].astype(str)

# Fill NaN values
all_topics_df = all_topics_df.fillna("None")

# Create combined documents dataframe
all_docs_df = pd.concat([cat_data['doc_df'] for cat_data in category_topics.values()])

# Write to Snowflake
print("Writing topic info to Snowflake...")
session.write_pandas(
    all_topics_df,
    table_name="TENSION_CATEGORIES_TOPIC_INFO",
    database="TRANSFORM_ENGCA_DEV",
    schema="DBT_MMARKS_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)

print("Writing documents with topics to Snowflake...")
session.write_pandas(
    all_docs_df,
    table_name="TENSION_CATEGORIES_DOCS_WITH_TOPICS",
    database="TRANSFORM_ENGCA_DEV",
    schema="DBT_MMARKS_ETHELO",
    overwrite=True,
    quote_identifiers=False,
    auto_create_table=True
)

print("Topic modeling complete!")

# Apply Topic Labels

In [ ]:
-- Cell 2: SQL to generate LLM-based topic descriptions using Snowflake Cortex
CREATE OR REPLACE TABLE TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_TOPIC_DESCRIPTIONS AS


WITH topic_comment_agg AS (
    SELECT 
        a.TOPIC,
        a.CATEGORY,
        COUNT(*) AS n,
        LISTAGG(SUBSTRING(CONTENT, 1, 500), '; ') AS topic_comments
        ,b.representation
    from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_DOCS_WITH_TOPICS as a
    left join TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_TOPIC_INFO as b on a.topic = b.topic
    where a.original_topic >= 0
    GROUP BY a.TOPIC, a.CATEGORY,REPRESENTATION
),

topic_desc_raw AS (
    SELECT
        topic,
        category,
        n,
        LENGTH(topic_comments) AS n_char,
        topic_comments,
        SNOWFLAKE.CORTEX.COMPLETE('llama4-maverick',
            CONCAT('''The following is a semi-colon separated list of comments from people impacted by recent wildfires in Los Angeles county (the Eaton Fire in Altadena and the Palisades fire in Pacific Palisades). For these comments, please provide a brief topic title. The keywords associated with these comments and this topic are:''',representation, '''. The topic title should be between 2 and 5 words and should be specific and descriptive of the throughline concern addressed by the comments. These comments are all on one side of a three-dimensional debate. The topic title should state what side they are on. It is crucial that the title be representative of every single comment you are provided. Your response should contain the 2-5 word topic title and nothing else. Only use alphanumeric characters and spaces. No quotes.  <comments>''', 
            topic_comments, '</comments>')) AS desc_raw
    FROM topic_comment_agg
)

SELECT
    topic,
    category,
    n,
    desc_raw AS title
FROM topic_desc_raw;

select * from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_TOPIC_DESCRIPTIONS

# CSV for Web Visuals

In [ ]:
select c.COMMENT_ID
    , c.TARGET
    , c.CONTENT
    , c.tension_category
    , llm_summary
    , t.UMAP_1
    , t.UMAP_2
    ,case when topic = 'Build_11_4' then 'ADU considerations'
        when topic = 'Build_11_3' then 'Soil testing and environmental recovery'
        when topic = 'Build_11_2' then 'Streamline rebuilding process'
        when topic = 'Build_11_1' then 'Bury power lines'
        when topic = 'Build_11_0' then 'Rebuild with fire resilience'
        
        when topic = 'Densi_0_2' then 'ADU considerations'
        when topic = 'Densi_0_1' then 'Evacuation routes'
        when topic = 'Densi_0_0' then 'Re-zoning areas for rebuilding'
        
        when topic = 'Indiv_17_2' then 'Local community accountability'
        when topic = 'Indiv_17_1' then 'Government accountability'
        when topic = 'Indiv_17_0' then 'Fire hardening and resilience'
        
        when topic = 'Publi_3_3' then 'Rebuilding costs accountability'
        when topic = 'Publi_3_2' then 'Infrastructure maintenance accountability'
        when topic = 'Publi_3_1' then 'Insurance industry reform'
        when topic = 'Publi_3_0' then 'Bury power lines'
        else SUBCATEGORY 
        end as SUBCATEGORY
from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_COMMENTS_WITH_CATS_AND_EMBEDDINGS as c
left join (
    select COMMENT_ID, a.TOPIC, UMAP_1, UMAP_2, ifnull(TITLE, 'None') as SUBCATEGORY ,a.CATEGORY
    from TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_DOCS_WITH_TOPICS as a
    left join TRANSFORM_ENGCA_DEV.DBT_MMARKS_ETHELO.TENSION_CATEGORIES_TOPIC_DESCRIPTIONS as b 
    on a.TOPIC = b.topic
) as t on c.COMMENT_ID = t.COMMENT_ID and c.tension_category = t.CATEGORY

# All Agenda Setting Comments

In [ ]:
with comments as (
    select * from transform_engca_prd.analytics.stg_comments
),

categories as (
    select *
    from transform_engca_dev.dbt_mmarks_ethelo.tension_comments_with_cats_and_embeddings
    where tension_category != 'Financial Aid Distribution'
),

to_publish as (
    select
        comments.comment_id as "comment_id",
        any_value(comments.content) as "comment",
        any_value(comments.target) as "topic",
        listagg(
            case (categories.tension_category)
                when 'Build Faster v. Build Better'
                    then 'Rebuild quickly or rebuild for resilience'
                when 'Public v Private'
                    then 'Public or private sector responsibility for utilities and infrastructure'
                when 'Individual v. Collective Mitigation'
                    then 'Individual or collective responsibility for fire prevention'
                when 'Density v. Safety'
                    then 'Making space for more residents, or limiting housing density'
                else null
            end,
            ';'
        ) as "opportunity_area",
    from comments
    left join categories
    on comments.comment_id = categories.comment_id
    group by comments.comment_id
    order by comments.comment_id desc
)

select * from to_publish;